In [0]:
%sql

DESCRIBE bakery_pos_develop.silver.pos_order_line

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_pos_order_line AS
SELECT
    line.pos_order_line_id,
    CAST(date_format(pos_order.date_order, 'yyyyMMdd') AS INT) AS date_sk,
    user.user_sk,
    customer.customer_sk,
    order.pos_order_sk,
    session.session_sk,
    product.product_sk,
    line.price_subtotal,
    line.price_subtotal_incl,
    line.price_unit,
    line.qty,
    line.total_cost,
    line.discount_amount,
    line.marging_amount,
    line.created_at,
    line.updated_at,
    line.is_updated,
    line.is_deleted
FROM bakery_pos_develop.silver.pos_order_line line
INNER JOIN
bakery_pos_develop.silver.pos_order pos_order
ON
line.pos_order_id = pos_order.pos_order_id
INNER JOIN
bakery_pos_develop.gold.dim_res_users user
ON
pos_order.user_id = user.user_id
INNER JOIN
bakery_pos_develop.gold.dim_res_partner customer
ON
pos_order.partner_id = customer.customer_id
INNER JOIN 
bakery_pos_develop.gold.dim_pos_order order
ON
line.pos_order_id = order.pos_order_id
INNER JOIN
bakery_pos_develop.gold.dim_pos_session session
ON
pos_order.session_id = session.session_id
INNER JOIN
bakery_pos_develop.gold.dim_product_product product
ON
line.product_id = product.product_id
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY line.pos_order_line_id
    ORDER BY line.updated_at DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.fact_pos_order_line (
    pos_order_line_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    pos_order_line_id BIGINT,
    date_sk INT,
    user_sk BIGINT,
    customer_sk BIGINT,
    pos_order_sk BIGINT,
    session_sk BIGINT,
    product_sk BIGINT,
    price_subtotal DECIMAL (10, 2),
    price_subtotal_incl DECIMAL(10, 2),
    price_unit DECIMAL(10, 2),
    qty DECIMAL(10, 2),
    total_cost DECIMAL(10, 2),
    discount_amount DECIMAL(10, 2),
    marging_amount DECIMAL(10, 2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (pos_order_line_sk),
    FOREIGN KEY (date_sk) REFERENCES bakery_pos_develop.gold.dim_date(date_sk),
    FOREIGN KEY (user_sk) REFERENCES bakery_pos_develop.gold.dim_res_users (user_sk),
    FOREIGN KEY (customer_sk) REFERENCES bakery_pos_develop.gold.dim_res_partner (customer_sk),
    FOREIGN KEY (pos_order_sk) REFERENCES bakery_pos_develop.gold.dim_pos_order (pos_order_sk),
    FOREIGN KEY (session_sk) REFERENCES bakery_pos_develop.gold.dim_pos_session (session_sk),
    FOREIGN KEY (product_sk) REFERENCES bakery_pos_develop.gold.dim_product_product (product_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.fact_pos_order_line AS tgt
USING stg_pos_order_line AS src
ON tgt.pos_order_line_id = src.pos_order_line_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.date_sk = src.date_sk,
        tgt.user_sk = src.user_sk,
        tgt.customer_sk = src.customer_sk,
        tgt.pos_order_sk = src.pos_order_sk,
        tgt.session_sk = src.session_sk,
        tgt.product_sk = src.product_sk,
        tgt.price_subtotal = src.price_subtotal,
        tgt.price_subtotal_incl = src.price_subtotal_incl,
        tgt.price_unit = src.price_unit,
        tgt.qty = src.qty,
        tgt.total_cost = src.total_cost,
        tgt.discount_amount = src.discount_amount,
        tgt.marging_amount = src.marging_amount,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        pos_order_line_id,
        date_sk,
        user_sk,
        customer_sk,
        pos_order_sk,
        session_sk,
        product_sk,
        price_subtotal,
        price_subtotal_incl,
        price_unit,
        qty,
        total_cost,
        discount_amount,
        marging_amount,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.pos_order_line_id,
        src.date_sk,
        src.user_sk,
        src.customer_sk,
        src.pos_order_sk,
        src.session_sk,
        src.product_sk,
        src.price_subtotal,
        src.price_subtotal_incl,
        src.price_unit,
        src.qty,
        src.total_cost,
        src.discount_amount,
        src.marging_amount,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )